# Running Light Curves

In [195]:
# --- In your notebook ---
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import sys
import pandas as pd
from itertools import islice
from IPython.display import display
import re
#imports
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import sys
import os
from astropy.cosmology import Planck18 as cosmo
import ast
import george




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [197]:
%reload_ext autoreload

In [199]:


# If slsn_gp_lc.py lives in SLSNe_Metric/py_files/, add that parent to PYTHONPATH
repo_root = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric")
sys.path.insert(0, str(repo_root / "py_files"))  # contains slsn_gp_lc.py

# Import the module by name (matches filename without .py)
slsn_metric = importlib.import_module("local_SLSNe_metric")
importlib.reload(slsn_metric)
print("[CONFIG] Using Cristina's local MacBook setup")

# Handy aliases
CatalogInputs = slsn_metric.CatalogInputs
SLSN_LC       = slsn_metric.LC




[CONFIG] Using Cristina's local MacBook setup


In [201]:
per_event_csv_dir = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/per_event_files")
allparams_csv     = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/output/all_events/allparameter.csv")

# Use the same templates_file path you use elsewhere (from shared_utils.build_filenames, etc.)
# If you already computed this earlier as `templates_file`, reuse it; otherwise set explicitly:
templates_file = Path("/Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub/output/SLSNe/SLSNe_templates.pkl")

inputs = CatalogInputs(
    photometry_dir=per_event_csv_dir,
    params_table=pd.read_csv(allparams_csv),
)
SLSN_LC.from_catalog(
    inputs,
    filename_pattern="{name}.csv",
    save_to=templates_file,
    n_time=220, tpad_pre_days=5.0, tpad_post_days=160.0,
    filters_dir=Path("/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/SLSNe/slsne/ref_data/filters")
)



/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/py_files/local_SLSNe_metric.py:660: UserWarning: [filter read] .ipynb_checkpoints: IsADirectoryError(21, 'Is a directory')
  
/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/py_files/local_SLSNe_metric.py:660: UserWarning: [filter read] 2MASS_2MASS.H_AB.dat: NameError("name 're' is not defined")
  
/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/py_files/local_SLSNe_metric.py:660: UserWarning: [filter read] 2MASS_2MASS.J_AB.dat: NameError("name 're' is not defined")
  
/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/py_files/local_SLSNe_metric.py:660: UserWarning: [filter read] 2MASS_2MASS.Ks_AB.dat: NameError("name 're' is not defined")
  
/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/py_files/local_SLSNe_metric.py:660: UserWarning: [filter read] Generic_Bessell.B_AB.dat: NameError("name 're' is not defined")
  
/Users/and

[INFO] Saved atomically to /Users/andradenebula/Documents/Research/Transient_Metrics/Multi_Transient_Metrics_Hub/output/SLSNe/SLSNe_templates.pkl


/Users/andradenebula/Documents/Research/Transient_Metrics/SLSNe_Metric/py_files/local_SLSNe_metric.py:443: UserWarning: [SCP06F6] unsupported filters / no λ_eff; skipping.
  


In [203]:


# --- Population & cadences (your usual knobs) ---
use_kcorrect=False
k_correct_type='powerlaw'
k_correct_arg=-0.75

testname=None
testname_metric_only=None

generate_new_pop = True
make_debug_plots = True

rate_density = 1e-8   # placeholder; you’ll set this later
z_min, z_max = 0.02, 2.0
d_min, d_max = None, None
gal_lat_cut  = None
use_extinction = True
t_start, t_end = 1, 3652

cadences = ['baseline_v5.0.0_10yrs', 'four_roll_v5.0.0_10yrs']
ignore_triples = True
clean_temp = True

# --- Build output paths with your helper ---
templates_file_, pop_file, df_file, storage_dir, summary_filename = shared_utils.build_filenames(
    rate_density=rate_density, z_min=z_min, z_max=z_max, d_min=d_min, d_max=d_max,
    science_case="SLSNe", testname=testname, testname_metric_only=testname_metric_only,
    ignore_triples=ignore_triples, use_extinction=use_extinction, use_kcorrect=use_kcorrect,
    base_dir=base_dir
)
# We already have a templates_file above; we’ll keep using that.


SLSNe_den_1e-08_d_None-None_Mpc_z_0.02-2.0_ext_True_kcor_False_None


In [ ]:
# # --- Build a single templates.pkl (only needs to run when new catalog is added) ---
# templates_file = Path(base_dir) / "Rubin_tests" / "SLSNe_catalog_templates.pkl"
# slsn_templates.build_slsn_templates(
#     per_event_dir     = per_event_csv_dir,
#     allparams_csv     = allparams_csv,
#     out_templates_pkl = templates_file,
#     n_time            = 220,
#     t_post_days       = 160.0,
#     min_points_for_fit= 8,
#     force_restframe   = True,
#     make_abs_mag      = True
# )



In [172]:
# after you’ve successfully imported the module:
# slsn_metric = importlib.import_module("py_files.local_SLSNe_metric")
# SLSN_LC = slsn_metric.LC

# 1) Build templates ONCE from your catalog (creates the pickle)
inputs = slsn_metric_mod.CatalogInputs(
    photometry_dir=per_event_csv_dir,
    params_table=pd.read_csv(allparams_csv),
)
LC.from_catalog(
    inputs,
    filename_pattern="{name}.csv",
    save_to=templates_file,     # <- write the pickle here
    n_time=220,
    tpad_pre_days=5.0,
    tpad_post_days=160.0,
)

# 2) Load templates for the pipeline
shared_lc_model = shared_utils.load_or_generate_templates(
    SLSN_LC,                     # <- pass the CLASS, not a string
    templates_file=templates_file,
    generate_new=False           # <- we already saved the pickle above
)

# 3) Optional quick plot
shared_utils.plot_template_lcs(
    templates_file,
    num=3, use_log_time=False, plot_overlap=True
)

AttributeError: 'str' object has no attribute 'CatalogInputs'

# Not yet

In [ ]:

# --- Population slicer (reuses your GRB machinery; templates are absolute-mag) ---
slicer = shared_utils.load_or_generate_population(
    use_extinction=use_extinction,
    lc_model=shared_lc_model,
    t_start=t_start, t_end=t_end,
    d_min=d_min, d_max=d_max, z_min=z_min, z_max=z_max,
    seed=42, num_lightcurves=None,  # read count from templates; not used
    gal_lat_cut=gal_lat_cut, rate_density=rate_density,
    pop_file=pop_file,
    generate_new=generate_new_pop, make_debug_plots=make_debug_plots,
    use_kcorrect=use_kcorrect, k_correct_type=k_correct_type, k_correct_arg=k_correct_arg
)

# --- Run metrics (detect placeholder here; add more later if desired) ---
multi_metrics = slsn_metric.get_multi_metrics(shared_lc_model, include=['detect'],
                                              use_extinction=use_extinction,
                                              use_kcorrect=use_kcorrect,
                                              k_correct_type=k_correct_type,
                                              k_correct_arg=k_correct_arg)

# optional: diagnostics knobs (same as your GRB code)
for m in multi_metrics:
    if hasattr(m, "diag_store"):
        m.diag_store = False
        m.diag_sample_rate = 0.01
        m.diag_per_event_cap = 30
        m.diag_min_snr = 3
        m.diag_max_mag = None

# quick detect run (produces ObsRecords_*.csv like GRB flow)
df_obs = shared_utils.run_detect(
    slsn_metric, slicer, cadences, shared_lc_model, db_dir, storage_dir, df_file,
    use_extinction=use_extinction, use_kcorrect=use_kcorrect,
    k_correct_type=k_correct_type, k_correct_arg=k_correct_arg,
    ignore_triples=ignore_triples, debug=True, plot=True, clean_temp=clean_temp
)

# multi-metric summary table (works even with only 'detect' present)
summary_df = shared_utils.run_multi_metrics(
    multi_metrics, slicer, cadences, shared_lc_model,
    db_dir, storage_dir, summary_filename=summary_filename,
    ignore_triples=False, plot=True, clean_temp=clean_temp, use_extinction=use_extinction
)
summary_df
